# CUDA Kernel 面试主线 · 第 5/12 课：LayerNorm：均值、方差与数值稳定

> 状态：**学习中（待提交）**  
> 本仓库采用逐课通过制。本课未通过前，不应直接进入下一课。

## 本课目标与完成标准

学完后你应能：实现 LayerNorm 统计量并识别 E[x²]-E[x]² 的消减风险。

通过必须同时满足：

- 独立补齐本课唯一的代码填空题，并通过给定检查；
- 三个问答题均说明因果链，而不是只报术语；
- 能指出至少一个正确性边界和一个性能取舍；
- 总分不低于 8/10，且没有一票否决级概念错误。

## 前置关系

- 课程前置：C/C++、线性代数、基本并行编程
- 本课在路线中的作用：LayerNorm 对一行计算均值和方差，再做仿射变换；不同 token 的行互相独立。

## 核心心智模型

### 1. 它是什么，解决什么问题

LayerNorm 对一行计算均值和方差，再做仿射变换；不同 token 的行互相独立。

### 2. 它如何工作

本基础版并行累加 sum 与 sumsq，归约后算 var=E[x²]-E[x]²，再同步广播 mean/inv_std。

### 3. 正确性条件与常见误区

浮点舍入可令 var 略小于 0，至少要 clamp；大均值小方差时应考虑 Welford。

### 4. 性能与工程取舍

Welford 更稳定但 combine 更复杂；融合前后算子可省内存但增加寄存器压力。

## 具体演示

x=[10000,10001] 时两个大平方相减得到很小方差，低精度会放大消减。

请在阅读后先合上这一节，用自己的语言复述“输入状态 → 中间状态 → 输出状态”，再做练习。

## 实践任务：唯一代码填空题

补齐方差表达式，并解释为何随后要 fmaxf。

规则：只能修改 `TODO`/`______` 所在位置；不要删除断言或放宽误差。代码注释说明了每个边界条件。

In [ ]:
%%writefile /tmp/05_layernorm.cu
#include <cuda_runtime.h>

namespace {

// 面试重点 1：warp 内 reduce 不需要 shared memory。
// 一个 warp 内 32 个线程可以通过 shuffle 指令直接交换寄存器里的值，
// 比写 shared memory 再同步更轻。
template<int BLOCK_SIZE>
__device__ __forceinline__ float warp_reduce_sum(float val) {
    #pragma unroll
    for (int offset = 16; offset > 0; offset >>= 1) {
        val += __shfl_down_sync(0xffffffff, val, offset);
    }
    return val;
}

// 面试重点 2：block reduce 通常分两层：
// 1. 每个 warp 先在寄存器里 reduce 出一个 partial sum。
// 2. 每个 warp 的 lane 0 把 partial sum 写入 shared memory。
// 3. warp 0 再把这些 warp partial sum reduce 成 block 级结果。
template<int BLOCK_SIZE>
__device__ __forceinline__ float block_reduce_sum(float val) {
    constexpr int NUM_WARPS = (BLOCK_SIZE + 31) / 32;
    __shared__ float smem[NUM_WARPS];

    int lane = threadIdx.x & 31;
    int warp = threadIdx.x >> 5;

    val = warp_reduce_sum<BLOCK_SIZE>(val);

    if (lane == 0) {
        smem[warp] = val;
    }
    __syncthreads();

    // 只有前 NUM_WARPS 个线程需要读取 shared memory。
    // 这些线程都在 warp 0 里，因此后续仍然可以用 warp reduce。
    val = (threadIdx.x < NUM_WARPS) ? smem[lane] : 0.0f;
    if (warp == 0) {
        val = warp_reduce_sum<BLOCK_SIZE>(val);
    }

    // 这个同步保证同一个 kernel 里连续调用 block_reduce_sum 时，
    // 下一次调用不会覆盖上一次还没读完的 smem。
    __syncthreads();
    return val;
}

// LayerNorm: 对每一行独立做归一化。
// input/output: [M, N] row-major
// gamma/beta: [N]
//
// 数学公式：
// mean = sum(x_i) / N
// var  = sum(x_i^2) / N - mean^2
// y_i  = (x_i - mean) / sqrt(var + eps) * gamma_i + beta_i
//
// 优化思路：
// 一个 block 负责一整行，block 内线程并行扫 N 个元素。
// 每个线程先做局部累加，再通过 warp reduce + smem 得到整行统计量。
template<int BLOCK_SIZE>
__global__ void layernorm_kernel(
    const float* __restrict__ input,
    const float* __restrict__ gamma,
    const float* __restrict__ beta,
    float* __restrict__ output,
    int M,
    int N,
    float eps
) {
    int row = blockIdx.x;
    int tid = threadIdx.x;
    if (row >= M) return;

    const float* row_in = input + row * N;
    float* row_out = output + row * N;

    float local_sum = 0.0f;
    float local_sq_sum = 0.0f;

    // 每个线程用 stride 方式处理多个列。
    // col 连续分配给 threadIdx.x，能保证同一轮访问时线程读连续地址，
    // 也就是 coalesced global memory load。
    for (int col = tid; col < N; col += BLOCK_SIZE) {
        float x = row_in[col];
        local_sum += x;
        local_sq_sum += x * x;
    }

    // 两个 reduce 分别得到 sum(x) 和 sum(x^2)。
    float sum = block_reduce_sum<BLOCK_SIZE>(local_sum);
    float sq_sum = block_reduce_sum<BLOCK_SIZE>(local_sq_sum);

    // mean 和 inv_std 是整行共享的标量。
    // 只让 tid 0 计算一次，然后放到 shared memory 广播给全 block。
    __shared__ float smem_mean;
    __shared__ float smem_inv_std;

    if (tid == 0) {
        float mean = sum / static_cast<float>(N);
        float var = ______;  // TODO: E[x^2] - E[x]^2

        // fmaxf 防止浮点舍入导致 var 变成很小的负数。
        smem_mean = mean;
        smem_inv_std = rsqrtf(fmaxf(var, 0.0f) + eps);
    }
    __syncthreads();

    float mean = smem_mean;
    float inv_std = smem_inv_std;

    // 第二次扫这一行，写最终归一化结果。
    // LayerNorm 至少需要先知道整行 mean/var，所以写输出通常是第二 pass。
    for (int col = tid; col < N; col += BLOCK_SIZE) {
        float y = (row_in[col] - mean) * inv_std;
        row_out[col] = y * gamma[col] + beta[col];
    }
}

} // namespace

// 简洁 launch wrapper：面试时可以先固定 BLOCK_SIZE=256，
// 后续再讨论如何根据 N 和 occupancy 调参。
void launch_layernorm(
    const float* input,
    const float* gamma,
    const float* beta,
    float* output,
    int M,
    int N,
    float eps,
    cudaStream_t stream
) {
    constexpr int BLOCK_SIZE = 256;
    layernorm_kernel<BLOCK_SIZE><<<M, BLOCK_SIZE, 0, stream>>>(
        input, gamma, beta, output, M, N, eps
    );
}


### 检查方法

有 CUDA 环境时执行 `nvcc -std=c++17 -c /tmp/05_layernorm.cu -o /tmp/05_layernorm.cu.o`；无 CUDA 环境时只做静态审查并登记待验证。

提交时请给出：补齐后的代码、实际运行输出（环境不可用时注明“仅静态审查”）以及对失败用例的解释。

### Q1

不要背定义：请从输入、状态变化和输出三个阶段解释“LayerNorm：均值、方差与数值稳定”的工作机制。

**你的答案：**


### Q2

方差出现 -1e-7 时直接 rsqrt 会发生什么？

**你的答案：**


### Q3

何时应把两矩统计改成 Welford，代价是什么？

**你的答案：**


## 评分与通过规则

- 代码 4 分：正常输入 2 分，边界输入 1 分，解释实现 1 分；
- Q1～Q3 各 2 分；
- 一票否决：结果碰巧正确但核心因果链错误、删除边界检查、把未运行结果说成实测。

需要提示时按四级机制请求：概念区域 → 具体方向 → 关键局部 → 完整答案。

## 参考资料

- [CUDA Programming Guide](https://docs.nvidia.com/cuda/cuda-programming-guide/)
- [CUDA Best Practices Guide](https://docs.nvidia.com/cuda/cuda-c-best-practices-guide/)

资料用于建立事实基线；面试回答仍需用自己的语言组织。